# Day 2-04：VLMに画像を見せる

画像と質問を入力し、答えが変わる条件を調べます。

## 実行前の準備

**`DRIVE_PROJECT`を自分のDrive内の教材フォルダに合わせてください。**
ローカルでは`notebooks/day2/`または教材ルートから実行します。ColabではGPUランタイムを選びます。

In [ ]:
#@title 環境設定（最初に実行）
from pathlib import Path
import runpy

###################################################################
# JKJ1A/へ移動．
DRIVE_PROJECT = Path("/content/drive/MyDrive/Colab Notebooks/JKJ1A")  # !!! 自分の環境に合わせて変更
LOCAL_PROJECT = (
    Path.cwd()
    if (Path.cwd() / "jkj1a/launcher.py").is_file()
    else Path.cwd().parent.parent
)
###################################################################

try:
    from google.colab import drive
    ON_COLAB = True
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
    project_hint = DRIVE_PROJECT
except ImportError:
    ON_COLAB = False
    project_hint = LOCAL_PROJECT

_context = runpy.run_path(str(project_hint / "jkj1a/launcher.py"))
ON_COLAB = _context["ON_COLAB"]
project_root = _context["project_root"]
print("実行場所:", "Colab" if ON_COLAB else "ローカル")
print("作業フォルダ:", project_root)

## 1. 小型VLM

分類器が固定されたクラスを選ぶのに対し、VLMは画像と文章から回答を生成します。ここでは英語の質問を使います。モデルの読み込みは最初の1回だけ行います。

In [ ]:
from jkj1a.vision_language import MiniVLM

vlm = MiniVLM(model_id="HuggingFaceTB/SmolVLM-500M-Instruct")

## 2. 数を数える

青い円の個数をプログラムで決めます。まず、文字を追加していない画像への回答を確認します。回答の生成ではサンプリングを使いません。

In [ ]:
from jkj1a.vision_language import counting_image

n = 5
image = counting_image(n=n, layout="grid", seed=0)
question = "How many blue circles are visible? Answer with only the number."

display(image)
answer = vlm.ask(image, question)
print("正解:", n, "回答:", answer)

## 3. 配置を変える

個数を保ち、円の位置をずらします。各画像に対する回答をそのまま記録してください。

In [ ]:
# 条件ごとの回答を記録する
observations = []

for layout in ("grid", "jitter"):

    for n in (3, 5, 8):
        picture = counting_image(n=n, layout=layout, seed=0)
        response = vlm.ask(picture, question)
        observations.append({"layout": layout, "expected": n, "answer": response})

        display(picture)
        print(observations[-1], flush=True)

## 4. 画像に文字を追加する

図形を隠さずに文字を追加します。画像サイズの変化の影響もあるため、正しい文字・誤った文字・無関係な文字の条件を比較します。図形の個数は全条件で同じです。

In [ ]:
from jkj1a.vision_language import add_text

base = counting_image(n=5)

for text in ("5 circles", "3 circles", "blue circles"):
    picture = add_text(base, text, font_size=32)
    response = vlm.ask(picture, question)

    display(picture)
    print(text, "->", response, flush=True)

## 5. 記録を保存する

画像・質問・正解・モデルの回答・実行条件を保存します。別の実験では`name`を変えてください。

In [ ]:
from jkj1a.vision_language import save_observation

picture = counting_image(n=5, layout="jitter", seed=1)
response = vlm.ask(picture, question)

# 入力画像・質問・回答・正解をまとめて保存する
path = save_observation(vlm, picture, question, response,
                        name="count-five-jitter-seed1", expected=5,
                        condition={"layout": "jitter", "seed": 1, "n": 5})
print(path)

## 演習

1. 個数を固定して配置や円の大きさを変え、正解する条件と間違える条件を記録してください。
2. 文字を入れないと正解する画像を選び、追加する文字の内容を変えて回答を比較してください。
3. 物体の一部を増減した画像を用意してください。元画像と編集画像に同じ質問を与え、実際に見える個数と回答を記録してください。

画像は`Image.open`で読み込めます。たとえば足を数える場合は、隠れている足を含めず「画像内に見える足の本数」を正解とします。正解が曖昧な画像は比較から外してください。

In [ ]:
from PIL import Image
# original = Image.open("data/day2/original.png").convert("RGB")
# edited = Image.open("data/day2/edited.png").convert("RGB")
# question = "How many legs are visible in this image? Answer with only the number."
# print(vlm.ask(original, question))
# print(vlm.ask(edited, question))